# 03 — Memory, Device, Views, and Autograd Fundamentals (Sections 6, 7, 8)

Covers:
- **Section 6:** Tensor views vs copies, contiguity/non-contiguous tensors, memory layout, in-place operations, CPU/GPU transfers, device consistency, basic GPU memory awareness (coding 1-10).
- **Section 7:** Derivatives, partial derivatives, computational graphs, forward computation, gradient tracking, `requires_grad`, leaf tensors, differentiable operations (coding 1-8).
- **Section 8:** Chain rule, backward propagation, gradient calculation/accumulation, `.backward()`, `.grad`, `torch.no_grad()`, `torch.detach()`, gradient clearing (coding 1-10).


In [1]:
import numpy as np
import torch

torch.manual_seed(0)
np.random.seed(0)

def get_device():
  '''Return preferred computation device (CUDA -> MPS -> CPU).'''
  if torch.cuda.is_available():
    return torch.device('cuda')
  if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    return torch.device('mps')
  return torch.device('cpu')

device = get_device()
print('Selected device:', device)
print('CUDA available:', torch.cuda.is_available())
print('MPS available:', hasattr(torch.backends, 'mps') and torch.backends.mps.is_available())


Selected device: cpu
CUDA available: False
MPS available: False


## Section 6 — Memory, Views, Contiguity, and Devices

### 1. Create a tensor and generate a view of it
### 2. Change values through a view and inspect the original tensor
### 3. Compare a view with an independent copy

In [2]:
base = torch.arange(6, dtype=torch.float32).reshape(2, 3)
view = base.view(3, 2)  # view (shares storage when contiguous/compatible)
print('base:\n', base)
print('view:\n', view)
print('shares storage (view):', view.data_ptr() == base.data_ptr() or base._is_view() is False)

view[0, 0] = 999.0
print('\nAfter modifying view[0,0]=999:')
print('base:\n', base)
print('view:\n', view)

base2 = torch.arange(6, dtype=torch.float32).reshape(2, 3)
copy = base2.clone().reshape(3, 2)
copy[0, 0] = -1.0
print('\nIndependent copy case:')
print('base2 unchanged:\n', base2)
print('copy modified:\n', copy)
print('copy shares storage with base2:', copy.data_ptr() == base2.data_ptr())


base:
 tensor([[0., 1., 2.],
        [3., 4., 5.]])
view:
 tensor([[0., 1.],
        [2., 3.],
        [4., 5.]])
shares storage (view): True

After modifying view[0,0]=999:
base:
 tensor([[999.,   1.,   2.],
        [  3.,   4.,   5.]])
view:
 tensor([[999.,   1.],
        [  2.,   3.],
        [  4.,   5.]])

Independent copy case:
base2 unchanged:
 tensor([[0., 1., 2.],
        [3., 4., 5.]])
copy modified:
 tensor([[-1.,  1.],
        [ 2.,  3.],
        [ 4.,  5.]])
copy shares storage with base2: False


### 4. Transpose a tensor and inspect its contiguity
### 5. Experiment with reshaping a non-contiguous tensor
### 6. Compare contiguous and non-contiguous tensor behavior

In [3]:
t = torch.arange(12).reshape(3, 4)
tt = t.transpose(0, 1)
print('t shape:', tuple(t.shape), 'contiguous:', t.is_contiguous(), 'strides:', t.stride())
print('t.T shape:', tuple(tt.shape), 'contiguous:', tt.is_contiguous(), 'strides:', tt.stride())

print('\nReshaping non-contiguous tensor:')
print('Predicted: reshape may work by returning a copy when needed; view may fail on non-contiguous layout.')
try:
  r = tt.reshape(-1)
  print('reshape(-1) success, shape:', tuple(r.shape), 'contiguous:', r.is_contiguous())
except Exception as e:
  print('reshape failed:', type(e).__name__, str(e))
try:
  v = tt.view(-1)
  print('view(-1) unexpected success', tuple(v.shape))
except Exception as e:
  print('view(-1) exception type:', type(e).__name__)
  print('view(-1) exception message:', str(e))

tt_contig = tt.contiguous()
print('\nAfter contiguous():', tt_contig.is_contiguous(), tt_contig.stride())
print('view(-1) on contiguous copy works:', tuple(tt_contig.view(-1).shape))


t shape: (3, 4) contiguous: True strides: (4, 1)
t.T shape: (4, 3) contiguous: False strides: (1, 4)

Reshaping non-contiguous tensor:
Predicted: reshape may work by returning a copy when needed; view may fail on non-contiguous layout.
reshape(-1) success, shape: (12,) contiguous: True
view(-1) exception type: RuntimeError
view(-1) exception message: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

After contiguous(): True (3, 1)
view(-1) on contiguous copy works: (12,)


### 7. Move a tensor between CPU and GPU if a GPU is available
### 8. Check whether two tensors are on compatible devices
### 9. Observe what happens when you attempt an operation involving incompatible devices

In [4]:
cpu_t = torch.tensor([1., 2., 3.], device='cpu')
print('cpu_t device:', cpu_t.device)

if torch.cuda.is_available():
  gpu_t = cpu_t.to('cuda')
  back_cpu = gpu_t.to('cpu')
  print('Moved to CUDA:', gpu_t.device)
  print('Moved back to CPU:', back_cpu.device)
  print('Compatible devices (gpu_t, gpu_t+1):', gpu_t.device == (gpu_t + 1).device)
  print('Compatible with CPU tensor?', cpu_t.device == gpu_t.device)
  print('\nIncompatible-device operation demo (real cross-device):')
  print('Predicted: FAIL - tensors on cpu and cuda cannot be combined directly.')
  try:
    _ = cpu_t + gpu_t
    print('Unexpected success')
  except Exception as e:
    print('Actual exception type:', type(e).__name__)
    print('Actual exception message:', str(e))
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
  mps_t = cpu_t.to('mps')
  print('Moved to MPS:', mps_t.device)
  print('Compatible with CPU tensor?', cpu_t.device == mps_t.device)
  print('\nIncompatible-device operation demo (real cross-device):')
  print('Predicted: FAIL - tensors on cpu and mps cannot be combined directly.')
  try:
    _ = cpu_t + mps_t
    print('Unexpected success')
  except Exception as e:
    print('Actual exception type:', type(e).__name__)
    print('Actual exception message:', str(e))
else:
  print('No accelerator device available (CPU-only environment).')
  print('\nIncompatible/unavailable-device operation demo (safe, caught):')
  print('Predicted: FAIL - moving to unavailable CUDA device should raise an error in this environment.')
  print('Rationale: demonstrates device compatibility/availability handling when only CPU is present.')
  try:
    _ = torch.ones(2).to('cuda')
    print('Unexpected success (CUDA unexpectedly available?)')
  except Exception as e:
    print('Actual exception type:', type(e).__name__)
    print('Actual exception message:', str(e))

print('\nBasic GPU memory awareness:')
if torch.cuda.is_available():
  print('CUDA memory allocated:', torch.cuda.memory_allocated())
  print('CUDA memory reserved:', torch.cuda.memory_reserved())
else:
  print('CUDA not available; GPU memory stats not applicable on this device.')


cpu_t device: cpu
No accelerator device available (CPU-only environment).

Incompatible/unavailable-device operation demo (safe, caught):
Predicted: FAIL - moving to unavailable CUDA device should raise an error in this environment.
Rationale: demonstrates device compatibility/availability handling when only CPU is present.
Actual exception type: RuntimeError
Actual exception message: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

Basic GPU memory awareness:
CUDA not available; GPU memory stats not applicable on this device.


### 10. Investigate the consequences of modifying a tensor in-place

In [5]:
x = torch.tensor([1., 2., 3.], requires_grad=True)
y = x * 2
print('y before in-place on x:', y)

# In-place modification on tensors involved in autograd can be unsafe/error-prone
a = torch.tensor([1., 2., 3.])
b = a
a.add_(5)  # in-place
print('In-place add_ modifies shared storage:')
print('a:', a, 'b (same object/storage view):', b)

# Demonstrate autograd-related in-place issue safely
w = torch.tensor([1., 2., 3.], requires_grad=True)
z = w * 3
print('\nIn-place on a tensor that is part of/needed by graph can raise errors:')
print('Predicted: potential runtime error when in-place op breaks saved tensor/version needed for backward.')
try:
  w.add_(1.0)
  out = z.sum()
  out.backward()
  print('Unexpected success; grad:', w.grad)
except Exception as e:
  print('Actual exception type:', type(e).__name__)
  print('Actual exception message:', str(e))


y before in-place on x: tensor([2., 4., 6.], grad_fn=<MulBackward0>)
In-place add_ modifies shared storage:
a: tensor([6., 7., 8.]) b (same object/storage view): tensor([6., 7., 8.])

In-place on a tensor that is part of/needed by graph can raise errors:
Predicted: potential runtime error when in-place op breaks saved tensor/version needed for backward.
Actual exception type: RuntimeError
Actual exception message: a leaf Variable that requires grad is being used in an in-place operation.


## Section 7 — Derivatives and Autograd Fundamentals
### 1. Create a scalar tensor with gradient tracking enabled
### 2. Perform several mathematical operations on it
### 3. Create a computational graph involving multiple tensors


In [6]:
x = torch.tensor(2.0, requires_grad=True)  # scalar with grad tracking
y = x ** 2 + 3 * x + 1
print('x:', x, 'requires_grad:', x.requires_grad)
print('y = x^2 + 3x + 1 =', y)

a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)
c = a * b + a ** 2
d = c / b + b
print('\nMulti-tensor graph: a=', a.item(), 'b=', b.item(), 'c=', c.item(), 'd=', d.item())


x: tensor(2., requires_grad=True) requires_grad: True
y = x^2 + 3x + 1 = tensor(11., grad_fn=<AddBackward0>)

Multi-tensor graph: a= 2.0 b= 3.0 c= 10.0 d= 6.3333330154418945


### 4. Inspect which tensors require gradients
### 5. Identify leaf and non-leaf tensors
### 6. Determine which operations are tracked by autograd

In [7]:
for name, t in [('x', x), ('y', y), ('a', a), ('b', b), ('c', c), ('d', d)]:
  print(name, 'value=', t.detach().item() if t.numel()==1 else t, 'requires_grad=', t.requires_grad, 'is_leaf=', t.is_leaf, 'grad_fn=', t.grad_fn)

print('\nTracked operations create grad_fn on non-leaf results (e.g. y,c,d). Leaf tensors created by user have no grad_fn but can require grads.')


x value= 2.0 requires_grad= True is_leaf= True grad_fn= None
y value= 11.0 requires_grad= True is_leaf= False grad_fn= <AddBackward0 object at 0x7f602c790a00>
a value= 2.0 requires_grad= True is_leaf= True grad_fn= None
b value= 3.0 requires_grad= True is_leaf= True grad_fn= None
c value= 10.0 requires_grad= True is_leaf= False grad_fn= <AddBackward0 object at 0x7f602bf2b730>
d value= 6.3333330154418945 requires_grad= True is_leaf= False grad_fn= <AddBackward0 object at 0x7f602bf2b760>

Tracked operations create grad_fn on non-leaf results (e.g. y,c,d). Leaf tensors created by user have no grad_fn but can require grads.


### 7. Observe what happens when gradient tracking is disabled

In [8]:
p = torch.tensor(2.0, requires_grad=True)
with torch.no_grad():
  q = p * 5 + 1
print('Inside no_grad: q.requires_grad=', q.requires_grad, 'grad_fn=', q.grad_fn)

r = torch.tensor(2.0, requires_grad=True)
s = (r * 2).detach() + 1
print('Detached path: s.requires_grad=', s.requires_grad, 'grad_fn=', s.grad_fn)

print('\nPredicted: backward on values not connected to tracked graph will fail.')
try:
  q.backward()
  print('Unexpected success')
except Exception as e:
  print('Actual exception type:', type(e).__name__)
  print('Actual exception message:', str(e))


Inside no_grad: q.requires_grad= False grad_fn= None
Detached path: s.requires_grad= False grad_fn= None

Predicted: backward on values not connected to tracked graph will fail.
Actual exception type: RuntimeError
Actual exception message: element 0 of tensors does not require grad and does not have a grad_fn


### 8. Draw the computational graph for three simple mathematical expressions

1. `y1 = x^2 + 3`
2. `y2 = a*b + b`
3. `y3 = (u+v) * w`

```mermaid
flowchart LR
  subgraph Expr1
    x1[x] --> s1[square]
    s1 --> add1[+ 3]
    c1[3] --> add1
    add1 --> y1[y1]
  end
  subgraph Expr2
    a2[a] --> m2[mul]
    b2[b] --> m2
    m2 --> add2[+ b]
    b2 --> add2
    add2 --> y2[y2]
  end
  subgraph Expr3
    u3[u] --> add3[+]
    v3[v] --> add3
    add3 --> m3[mul]
    w3[w] --> m3
    m3 --> y3[y3]
  end
```


## Section 8 — Backpropagation and Gradient Mechanics
### 1. Calculate gradients for simple polynomial expressions
### 2. Calculate gradients for expressions involving two independent variables
### 3. Construct a computational graph involving addition and multiplication
### 4. Calculate gradients for a nested mathematical expression
### 5. Inspect the gradients of individual tensors


In [9]:
# 1. polynomial y = 3x^3 - 2x^2 + x - 5, dy/dx = 9x^2 - 4x + 1
x = torch.tensor(2.0, requires_grad=True)
y = 3*x**3 - 2*x**2 + x - 5
y.backward()
print('1) x=2, y=', y.item(), 'dy/dx=', x.grad.item(), 'expected=', 9*2**2 - 4*2 + 1)

# 2. two independent variables f = x^2*y + y^3
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)
f = x**2 * y + y**3
f.backward()
print('2) df/dx=', x.grad.item(), 'expected=', 2*2*3)
print('   df/dy=', y.grad.item(), 'expected=', 2**2 + 3*3**2)

# 3. addition and multiplication graph
a = torch.tensor(1.5, requires_grad=True)
b = torch.tensor(-2.0, requires_grad=True)
c = (a + b) * (a * b)
c.backward()
print('3) c=', c.item(), 'dc/da=', a.grad.item(), 'dc/db=', b.grad.item())

# 4. nested expression
x = torch.tensor(1.0, requires_grad=True)
z = ((x + 1)**2 * (2*x - 3) + 4)**2
z.backward()
print('4) nested z=', z.item(), 'dz/dx=', x.grad.item())

# 5. inspect gradients (stored in .grad of leaf tensors)
p = torch.tensor(2.0, requires_grad=True)
q = torch.tensor(4.0, requires_grad=True)
r = p*q + p**2
r.backward()
print('5) p.grad=', p.grad, 'q.grad=', q.grad)


1) x=2, y= 13.0 dy/dx= 29.0 expected= 29
2) df/dx= 12.0 expected= 12
   df/dy= 31.0 expected= 31
3) c= 1.5 dc/da= -2.0 dc/db= -3.75
4) nested z= 0.0 dz/dx= 0.0
5) p.grad= tensor(8.) q.grad= tensor(2.)


### 6. Perform backward propagation more than once and observe gradient accumulation
### 7. Clear accumulated gradients and repeat the experiment

In [10]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2
y.backward()
print('First backward, grad:', x.grad.item())
# backward again on same graph would fail if graph freed; use retained graph example via fresh forward
y = x**2
y.backward()
print('After second backward without clearing (new forward, accumulated):', x.grad.item())
print('Expected accumulation: 4 + 4 = 8')

# clear and repeat
x.grad.zero_()
y = x**2
y.backward()
print('After zero_() and backward:', x.grad.item())


First backward, grad: 4.0
After second backward without clearing (new forward, accumulated): 8.0
Expected accumulation: 4 + 4 = 8
After zero_() and backward: 4.0


### 8. Compare gradient-tracked and gradient-disabled calculations
### 9. Detach a tensor from its computation graph and observe the effect

In [11]:
x = torch.tensor(3.0, requires_grad=True)
with torch.no_grad():
  y = x**2
print('no_grad result requires_grad:', y.requires_grad)

x = torch.tensor(3.0, requires_grad=True)
y = x**2
z = y.detach() * 2
print('detached tensor requires_grad:', z.requires_grad)
w = (x**2) * 2
w.backward()
print('Grad flows through original path, dw/dx:', x.grad.item())

x = torch.tensor(3.0, requires_grad=True)
y = x**2
z = y.detach()
out = z * x
out.backward()
print('Detached y blocks gradient through y; d(out)/dx from direct x only:', x.grad.item(), '(expected 9.0)')


no_grad result requires_grad: False
detached tensor requires_grad: False
Grad flows through original path, dw/dx: 12.0
Detached y blocks gradient through y; d(out)/dx from direct x only: 9.0 (expected 9.0)


### 10. Draw the forward and backward computational graphs for a simple expression

Example: `y = (x + 2) * 3`, let `a = x + 2`, `y = a * 3`.

```mermaid
flowchart LR
  subgraph Forward
    x[x] --> a[a = x + 2]
    two[2] --> a
    a --> y[y = a * 3]
    three[3] --> y
  end
  subgraph Backward
    gy[dy/dy = 1] --> ga[dy/da = 3]
    ga --> gx[dy/dx = 3]
  end
```


## Summary
Section 6 covered views/copies, contiguity, memory/layout, in-place ops, transfers/device compatibility and GPU memory awareness. Sections 7-8 covered autograd fundamentals, computational graphs, backward/grad mechanics, accumulation, no_grad, detach, and clearing.